# Provenance Agent — Data Workflow (Quickstart)

This notebook mirrors `../workflow.ipynb` for the **data** side. Given a notebook, the data workflow detects which variables hold the datasets used for analysis (via an LLM) and generates dataset citations (BibTeX).

**Functions covered:**
- `detect_datasets` / `detect_datasets_in_notebook` — LLM detection of dataset variables
- `build_retrieval_cell` — per-tool citation-retrieval source
- `filter_datasets` — narrow to all / one tool / one variable
- `inject_retrieval_cells` — append retrieval cells via nbformat
- `generate_data_workflow` — end-to-end: detect → filter → inject → write

## Setup

Add `src/` to the path. This notebook lives in `notebooks/testing/`, so `src/` is two levels up.

In [ ]:
import sys
sys.path.insert(0, '../../src')

from dataset_detection import detect_datasets, detect_datasets_in_notebook
from data_workflow import (
    build_retrieval_cell,
    filter_datasets,
    inject_retrieval_cells,
    generate_data_workflow,
)

## 1. `detect_datasets` — LLM detection of dataset variables

Detection is done by an LLM (Gemini via LangChain), not AST parsing. It traces the data flow to the **terminal** variable actually used for analysis and returns `[variable, tool]` pairs. Here it finds the filtered LiPDGraph result in the PaleoPCAlite example.

In [ ]:
pairs = detect_datasets_in_notebook('paleoPCAlite.ipynb')
print(pairs)

## 2. `build_retrieval_cell` — per-tool retrieval source

Each dataset gets a code cell that retrieves its BibTeX by reusing the already-loaded object in the live kernel (Approach C: `{var}.{method}`). LiPDGraph is special — its terminal variable is a DataFrame, so the cell converts it to a LiPD object first.

In [ ]:
for var, tool in [('D', 'PyLiPD'), ('ds', 'PyleoTUPS'), ('filtered_df2', 'LiPDGraph')]:
    print(f'# --- {tool} ---')
    print(build_retrieval_cell(var, tool))
    print()

## 3. `filter_datasets` — all / one tool / one variable

Supports returning every dataset citation, or filtering to a single tool or variable.

In [ ]:
sample = [['D', 'PyLiPD'], ['ds', 'PyleoTUPS'], ['filtered_df2', 'LiPDGraph']]
print('all:      ', filter_datasets(sample))
print('LiPDGraph:', filter_datasets(sample, tool='LiPDGraph'))
print('just ds:  ', filter_datasets(sample, variable='ds'))

## 4. `inject_retrieval_cells` — append cells with nbformat

Appends one retrieval code cell per detected dataset to a notebook node.

In [ ]:
import nbformat
demo = nbformat.v4.new_notebook()
inject_retrieval_cells(demo, [['filtered_df2', 'LiPDGraph']])
print(demo.cells[-1].source)

## 5. `generate_data_workflow` — end to end

Detects datasets, optionally filters, injects the retrieval cells, and writes the notebook back. Here we write to a **copy** so the original is untouched. Open that copy and run the injected cell **in its live kernel** (where `filtered_df2` exists) to print the BibTeX.

Pass `tool=` or `variable=` to cite only part of the notebook, e.g. `generate_data_workflow('paleoPCAlite.ipynb', tool='LiPDGraph')`.

In [ ]:
pairs = generate_data_workflow(
    'paleoPCAlite.ipynb',
    output_path='paleoPCAlite_with_citations.ipynb',
)
print('injected cells for:', pairs)